In [ ]:
%reset -f

import numpy as np
import scipy.optimize as SciOpt
from numpy import random as rnd
from numpy import linalg as LA
import matplotlib.pyplot as plt
import matplotlib
import time
from scipy.special import factorial

matplotlib.rcParams.update({'font.size': 18})

########################################################################
### Using Maximum Likelihood function as a parameter Estimator (MLE) ###
########################################################################

In [ ]:
### generate data for a poisson distribution "experiment", ###
### f(x|lambda) = lambda**x * exp(-lambda) / x!            ### 

NSAMPLES = 10000
LAMBDA   = 2.3
x        = rnd.poisson(LAMBDA,NSAMPLES); 

n,b,p = plt.hist(x,10,range=(-0.5,9.5))

plt.show()

In [ ]:
# define some functions

# returns probability density for RV and parameter

my_pdf = lambda data, lam : lam**data * np.exp(-lam)/factorial(data)

# returns log-likelihood term for RV and parameter

logL = lambda data, lam : np.log(my_pdf(data,lam))

# The cost function is just the sum of all log-likelihood
# terms, "-" sign because optimizer routines seek minimums

costfun = lambda lam: -sum(logL(x,lam)) 

In [ ]:
# Do the fit using a generic simplex algorithm

result  = SciOpt.minimize(costfun,2.1,method='Nelder-Mead')
print("The estimated parameter is: %8.5f" % float(result.x[0]))

In [ ]:
# Compare the analytical result for MLE

mle_analytical = np.mean(x)
std_analytical = np.std(x)/np.sqrt(NSAMPLES)

print("The estimate is : %8.5f +- %8.5f " % (mle_analytical,std_analytical))

In [ ]:
## now, repeat this experiment a bunch of times ###

NEXP = 1000            # number of experiments
accu = np.zeros(NEXP)  # data accumulator

for ii in np.linspace(1,NEXP,NEXP,dtype=int):

    # Generate NSAMPLES exponential RVs
    x = rnd.poisson(LAMBDA,NSAMPLES); 

    # Estimate Poisson parameter using MLE
    result = SciOpt.minimize(costfun, 2.1, method='Nelder-Mead')
    
    accu[ii-1] = result.x[0]
    
print('Estimator standard deviation is: %10.5f' % float(np.std(accu)))

In [ ]:
# Plot the results

fig = plt.figure(figsize=(10,5))

n,b,p = plt.hist(accu,100,range=(1.5,2.5))

xminus = np.mean(accu) - np.std(accu)
xplus  = np.mean(accu) + np.std(accu)

plt.plot([xminus,xminus],[0.0,1000.0],color='r')
plt.plot([xplus, xplus], [0.0,1000.0],color='r')
plt.axis([1.5,2.5,0.0,270.0])
plt.xlabel("MLE Estimate for $\\lambda$")
plt.ylabel("Counts at this value")
plt.show()